# GIAI ĐOẠN 4: WALKTHROUGH ĐỘNG CƠ HYBRID RAG PHÁP LÝ VAS & SOI THUYẾT MINH BCTC THẬT
**Dự án:** FinAudit AI — FinForensics (SE-Hackathon Challenge 2026)  
**Nguồn dữ liệu thật 100%:**
1. Kho văn bản pháp lý chuẩn hóa (`legal_corpus/`): `VSA 240`, `TT 99/2025/TT-BTC` (hiệu lực 01/01/2026 thay thế `TT 200/2014`), `TT 48/2019/TT-BTC`, `VAS 14`, `VAS 26`, `Forensic Playbook`.
2. Thuyết minh `B09-DN` cấu trúc thật từ **Vietcap IQ API** (`section=NOTE`, 38 doanh nghiệp trọng điểm × 2018–2025, 166 chỉ tiêu `noc*`).
3. 4 bộ PDF BCTC Hợp nhất Kiểm toán 2025 thật từ cổng công bố công khai **Vietstock / HOSE** (`VNM`, `FPT`, `TDH`, `NVL`) + PDF gốc văn bản thuần đối chứng.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "stage_4_rag_legal_notes" else Path(r"d:/SEHackathon")
sys.path.insert(0, str(ROOT / "stage_4_rag_legal_notes"))

from importlib import import_module
rag_mod = import_module("04_hybrid_rag_engine")

summary = json.loads((ROOT / "data_store/processed/stage4_rag_benchmark_summary.json").read_text(encoding="utf-8"))
print("Thời gian thực thi GĐ 4:", summary["elapsed_seconds"], "giây")
print("Số văn bản pháp lý:", summary["legal_corpus_stats"]["n_documents"], "| Số đoạn Section-Aware:", summary["legal_corpus_stats"]["n_chunks"])

## 1. Kiểm định Đối sánh 6 Chiều giữa Thuyết minh B09-DN và B01-DN (38 Doanh nghiệp Thật)

In [ ]:
df_cc = pd.DataFrame(summary["b09_structured_audit"]["cross_check_b09_vs_b01"])
display(df_cc[["metric_pair", "n_comparable_firm_years", "exact_match_within_1pct", "exact_match_pct", "median_rel_diff_pct", "n_discrepancies"]])
print("\nCân đối nội tại B09-DN:", summary["b09_structured_audit"]["internal_consistency"])
print("\nSo sánh thực nghiệm B09-DN năm 2025 (Nhóm A Bluechip vs Nhóm Cảnh báo/Biến cố):")
display(pd.DataFrame(summary["b09_structured_audit"]["group_b09_empirical_2025"]).T)

## 2. Chẩn đoán 3 Tầng Chất lượng PDF Thật & Đối sánh Tam giác 3 Nguồn (NVL 2025)

In [ ]:
df_pdf = pd.DataFrame(summary["real_pdf_diagnostics"])
display(df_pdf[["file_name", "file_size_mb", "n_pages", "text_pages", "scan_or_empty_pages", "avg_chars_per_page", "ocr_noise_ratio_pct", "quality_status"]])

print("\nĐối sánh 3 nguồn NVL 2025 (B01 API vs B09 API vs PDF Kiểm toán Moore AISC):")
display(pd.DataFrame(summary["nvl_2025_triangulation"]["items"]))

## 3. Đánh giá Benchmark 4 Bộ Truy xuất (20 Câu Pháp lý + Anti-Hallucination + 6 Câu PDF Nhiễu OCR NVL 2025)

In [ ]:
print("=== A. Benchmark 20 câu hỏi Pháp lý VAS / VSA / TT 99/2025 / TT 48/2019 ===")
display(pd.DataFrame(summary["legal_benchmark_20q"]["comparison_by_mode"]).T)

print("\n=== B. Anti-Hallucination Refusal Test (5 câu Out-of-Scope) ===")
ah = summary["legal_benchmark_20q"]["anti_hallucination_test"]
print(f"Refusal Rate: {ah['refusal_rate_pct']}% ({ah['refused_correctly']}/{ah['n_out_of_scope_queries']}) | False Abstention trên 20 câu Gold: {ah['false_abstention_on_20_gold']}")
display(pd.DataFrame(ah["details"]))

print("\n=== C. Benchmark 6 câu hỏi thực chiến trên PDF Nhiễu OCR NVL 2025 (91 trang) ===")
display(pd.DataFrame(summary["nvl_pdf_benchmark_6q"]["comparison_by_mode"]).T)

## 4. Thử nghiệm Trực tiếp Cầu nối SHAP -> Hybrid RAG & Truy vấn Pháp lý Tương tác

In [ ]:
legal_chunks = rag_mod.load_legal_corpus_chunks(ROOT / "stage_4_rag_legal_notes/legal_corpus")
retriever = rag_mod.HybridRAGRetriever(legal_chunks, rrf_k=30)

test_queries = [
    "Thông tư 99/2025 đổi tên Bảng cân đối kế toán thành gì và hiệu lực khi nào?",
    "Nợ quá hạn từ 2 năm đến dưới 3 năm trích lập dự phòng thuế theo TT 48 là bao nhiêu %?",
    "Giá vàng SJC hôm nay tăng hay giảm?",
]
for q in test_queries:
    res = retriever.search(q, top_k=2, mode="hybrid_rrf", enable_abstention=True)
    print(f"\n[QUERY] {q}")
    if res["abstained"]:
        print("  -> [TỪ CHỐI CHỐNG ẢO GIÁC]:", res["abstention_reason"])
    else:
        for r in res["results"]:
            print(f"  -> #{r['rank']} {r['citation']} (RRF={r['score']:.4f}, TF-IDF={r['tfidf_wc']:.3f})")